# Lição 11 - Agente-para-Agente (A2A) Protocolo


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## O que é o Protocolo A2A?

O **Protocolo Agente-para-Agente (A2A)** é um standard aberto que permite que agentes de IA comuniquem,
descubram-se mutuamente e colaborem — mesmo quando são construídos em frameworks diferentes ou alojados
por serviços distintos.

Conceitos-chave:

- **Descoberta** – Os agentes publicam um *Cartão de Agente* que descreve as suas capacidades, tornando
  mais fácil para outros agentes (ou orquestradores) encontrarem o especialista certo para uma tarefa.
- **Troca de Mensagens** – Os agentes trocam mensagens estruturadas através de um protocolo comum, de modo que um
  pedido de um agente possa ser compreendido e cumprido por outro, independentemente da sua implementação interna.
- **Ciclo de Vida da Tarefa** – O A2A define estados tais como *submetida*, *a trabalhar*, *concluída* e
  *falhada*, dando ao orquestrador visibilidade total sobre como uma tarefa delegada está a progredir.

Nesta lição simulamos a colaboração ao estilo A2A, interligando três agentes de viagem especializados
num fluxo de trabalho onde cada agente contribui com a sua experiência e passa os resultados ao seguinte.


## Criando Agentes de Viagem Especializados


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Colaboração Multiagente através de Fluxo de Trabalho

Ligamos os três agentes num fluxo de trabalho sequencial que espelha a passagem de mensagens A2A:

1. **CurrencyExchangeAgent** recebe o pedido do utilizador e fornece orientação cambial.
2. **ActivityPlannerAgent** recebe o contexto enriquecido e acrescenta recomendações de atividades.
3. **TravelManagerAgent** sintetiza ambas as entradas num resumo final de viagem.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Compreender o A2A em Produção

Num ambiente de produção, o protocolo A2A desbloqueia cenários poderosos entre serviços:

| Capability | Description |
|---|---|
| **Interoperabilidade entre frameworks** | Um agente construído com um framework pode delegar tarefas a um agente construído com qualquer outro framework compatível com A2A, permitindo verdadeira interoperabilidade entre organizações. |
| **Fronteiras de serviço** | Os agentes podem residir em microserviços separados, regiões de cloud ou até em organizações diferentes, continuando a colaborar sem problemas. |
| **Descoberta dinâmica** | Um orquestrador pode consultar um registo de Agent Card em tempo de execução para encontrar o especialista mais adequado para uma dada subtarefa. |
| **Streaming e notificações push** | O A2A suporta Server-Sent Events (SSE) para atualizações de progresso em tempo real e notificações push para tarefas de longa duração. |

O fluxo de trabalho que construímos acima é uma versão simplificada, em processo, deste padrão. Numa implementação real, cada agente exporia um endpoint HTTP, publicaria um Agent Card e comunicaria via o A2A JSON-RPC protocol.


## Resumo

Nesta lição aprendeu:

1. **O que é o protocolo A2A** — um padrão aberto para descoberta, envio de mensagens,
   e gestão de tarefas entre agentes.
2. **Como criar agentes especializados** — um agente de Câmbio de Moeda, um agente Planeador de Atividades,
   e um orquestrador Gestor de Viagens.
3. **Como ligar agentes num fluxo de trabalho** — usando `WorkflowBuilder` para modelar a passagem sequencial
   de mensagens entre agentes.
4. **Como o A2A funciona em produção** — permitindo colaboração entre frameworks e serviços,
   com descoberta dinâmica e atualizações em streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Isenção de responsabilidade**:
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos por garantir a precisão, tenha em atenção que traduções automáticas podem conter erros ou imprecisões. O documento original, na sua língua nativa, deve ser considerado a fonte autoritativa. Para informações críticas, recomenda-se uma tradução profissional efetuada por um tradutor humano. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações erróneas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
